# Heckman 选择模型：样本选择偏误与两阶段估计

> 本讲义用于《金融数据分析与建模》课程。配套文件 `06_heckman_codes.ipynb` 用于生成 `./figs/` 中的配图和 `./data/heckman_loan_sim.csv`。本章主案例是企业银行贷款利率与贷款可得性。

Heckman 选择模型处理的核心问题是：结果变量并不是对所有样本都可观测，而是只在样本通过某个选择机制后才被观测。此时，直接在可观测子样本上估计结果方程，通常会把选择机制带来的非随机性遗漏掉，从而产生样本选择偏误。

## 引言：从银行贷款利率样本说起

考虑一个公司金融研究问题：企业特征如何影响银行贷款利率？

这类问题看起来可以直接估计如下回归：

$$
rate_i = x_i'\beta + u_i
$$

其中，$rate_i$ 是企业 $i$ 的银行贷款利率，$x_i$ 包括企业规模、杠杆率、盈利能力、现金持有、抵押品等变量。问题在于，$rate_i$ 不是对所有企业都可观测。只有企业获得银行贷款，或者在数据中披露了贷款合同或利息支出信息，研究者才可能观察到贷款利率。

因此，手头样本通常经历了一个选择过程：

1. 银行先决定是否发放贷款，企业才进入贷款利率样本；
2. 随后，在获得贷款企业中，银行与企业再形成具体贷款利率。

第一个过程对应 selection equation，第二个过程对应 outcome equation。

若获得贷款不是随机事件，那么只在获得贷款企业中做 OLS，本质上是在一个经过筛选的样本中估计融资成本决定因素。这个筛选机制可能与企业未观测风险、银企关系、银行信息优势等因素有关。若这些因素同时影响贷款利率，OLS 的误差项便不再满足条件均值为 0。

![选择机制与结果观测](./figs/limit_dep_heckman_fig01_selection_mechanism.png)

::: {.callout-note}
### 概念辨析：Heckman 与 Tobit 的观测机制不同

Tobit 模型中，未超过阈值的结果通常仍以边界值形式进入数据，例如 $y_i=0$。Heckman 选择模型中，未被选择样本的结果变量并不是 0，而是根本不可观测。例如，没有获得贷款的企业并没有可观测贷款利率。
:::

## 从政策背景和筛选机制理解样本选择

银行贷款不是一个简单的市场交易结果，而是一个带有信息筛选、风险评估和制度约束的审批过程。尤其在中小企业融资场景中，银行通常需要先判断企业是否满足基本授信条件，再决定贷款额度、期限、担保方式和利率。

在实际研究中，贷款利率样本可能来自以下几类筛选机制：

- 企业是否向银行提出融资申请；
- 银行是否愿意进入授信审查流程；
- 企业是否拥有足够抵押品或担保；
- 企业所在地区是否有足够银行网点或政银企对接窗口；
- 企业是否属于当地重点支持行业、园区或信用服务平台覆盖对象；
- 企业是否披露了贷款合同、利息支出或有息负债明细。

这些背景信息直接影响 selection 方程的设定。例如，本地银行网点密度、园区金融服务站、政银企对接平台、信用担保服务覆盖等变量，可能显著影响企业是否获得银行贷款。若这些变量主要影响“能否获得贷款”，但在控制企业风险和财务特征后不直接决定贷款合同利率，它们就可以作为选择方程中的排他性变量候选。

金融研究中的 Heckman 模型不应只是机械地写成两个方程。更关键的是：先把样本如何产生说清楚，再由样本产生机制推导 selection 方程；再把结果变量如何形成说清楚，由价格决定机制推导 outcome 方程。

::: {.callout-important}
### 避坑指南：排他性变量不能事后硬凑

排他性变量应来自样本选择过程本身。若某个变量既影响企业是否获得贷款，又直接影响贷款利率，就不宜只放在选择方程中。例如，抵押品可能提高贷款可得性，也可能降低贷款风险溢价；在这种情况下，更稳妥的做法通常是把它同时放入选择方程和结果方程。
:::

## 从因果推断角度理解 Heckman 模型

Heckman 选择模型可以放在更一般的因果推断框架中理解。普通 OLS 试图识别：

$$
E(y_i\mid x_i)
$$

但在存在样本选择时，研究者实际看到的是：

$$
E(y_i\mid x_i, s_i=1)
$$

其中，$s_i=1$ 表示结果变量被观测到。如果进入样本本身与结果方程中的未观测因素相关，即：

$$
s_i \not\perp u_i \mid x_i
$$

那么，选择变量 $s_i$ 携带了未观测信息。此时，直接在 $s_i=1$ 的样本中估计 $y_i$ 对 $x_i$ 的回归，本质上存在遗漏变量偏误。

在贷款利率案例中，$u_i$ 可以理解为研究者无法完全观察到、但银行在审批和定价中可能观察到的企业风险。如果银行更容易向低风险企业发放贷款，那么进入贷款利率样本的企业并非随机抽样。只在获得贷款企业中估计贷款利率方程，便可能低估或扭曲企业特征对融资成本的影响。

Heckman 两步法的思想可以概括为：

- 第一步，显式建模样本进入机制；
- 第二步，把选择机制中可推导出的非随机成分带回结果方程；
- 若修正项显著，说明进入结果样本的过程与结果方程误差项存在系统关联。

## Heckman 选择模型的基本设定

设潜在结果方程为：

$$
y_i^* = x_i'\beta + u_i
$$

设选择方程为：

$$
s_i^* = z_i'\gamma + v_i
$$

其中，$s_i^*$ 是潜在选择倾向，$s_i$ 是实际选择结果：

$$
s_i=
\begin{cases}
1, & s_i^*>0 \\
0, & s_i^*\leq 0
\end{cases}
$$

结果变量的观测规则为：

$$
y_i =
\begin{cases}
y_i^*, & s_i=1 \\
\text{missing}, & s_i=0
\end{cases}
$$

标准 Heckman 模型进一步假设：

$$
\begin{pmatrix}
u_i\\
v_i
\end{pmatrix}
\sim
N
\left[
\begin{pmatrix}
0\\
0
\end{pmatrix},
\begin{pmatrix}
\sigma_u^2 & \rho\sigma_u\\
\rho\sigma_u & 1
\end{pmatrix}
\right]
$$

这里，选择方程误差项 $v_i$ 的方差被标准化为 1，因为 Probit 模型只能识别相对尺度。若 $\rho\neq 0$，说明选择方程中的未观测因素与结果方程中的未观测因素相关，样本选择偏误就会出现。

## 贷款利率案例中的方程设定

本章案例把贷款利率视为 outcome，把是否获得贷款视为 selection。

结果方程为：

$$
rate_i^*
=
\beta_0
+
\beta_1 size_i
+
\beta_2 cash_i
+
\beta_3 lev_i
+
\beta_4 profit_i
+
\beta_5 collateral_i
+
u_i
$$

选择方程为：

$$
\begin{aligned}
loan_i^*
&=
\gamma_0
+
\gamma_1 size_i
+
\gamma_2 cash_i
+
\gamma_3 lev_i
+
\gamma_4 profit_i \\
&\quad
+
\gamma_5 collateral_i
+
\gamma_6 bank\_access_i
+
\gamma_7 credit\_window_i
+
v_i
\end{aligned}
$$

观测规则为：

$$
rate_i =
\begin{cases}
rate_i^*, & loan_i=1 \\
\text{missing}, & loan_i=0
\end{cases}
$$

变量含义如下：

| 变量 | 含义 | 进入方程 |
|---|---|---|
| `rate` | 银行贷款利率，仅对获得贷款企业可观测 | 结果变量 |
| `loan` | 是否获得银行贷款 | 选择变量 |
| `size` | 企业规模 | 两个方程 |
| `cash` | 现金持有 | 两个方程 |
| `lev` | 杠杆率 | 两个方程 |
| `profit` | 盈利能力 | 两个方程 |
| `collateral` | 抵押品充足程度 | 两个方程 |
| `bank_access` | 本地银行服务可得性 | 选择方程 |
| `credit_window` | 是否处于政银企对接窗口覆盖范围 | 选择方程 |

这里把 `bank_access` 和 `credit_window` 作为排他性变量候选。它们的经济含义是降低银企匹配成本、提高企业进入授信流程和获得贷款的概率；在控制企业风险、抵押品和财务特征后，它们不应直接决定单笔贷款合同利率。这个假设需要结合制度背景、数据口径和稳健性检验进行论证。

![排他性变量的设定逻辑](./figs/limit_dep_heckman_fig06_exclusion_logic.png)

::: {.callout-note}
### 经典文献中的方程设定思路

Heckman (1979) 把样本选择偏误表述为 specification error。核心并不是“样本变少了”，而是被观测样本的误差项条件均值不再为 0。Mroz (1987) 的劳动供给研究则提供了经典应用：工资只对就业女性可观测，因此选择方程解释是否就业，结果方程解释工资水平。

这两个例子对应同一设定思想：先定义结果变量何时可观测，再为这个“可观测性”建立选择方程；然后在可观测样本中建立结果方程。变量选择应服务于这个观测机制，而不是从同一组控制变量中机械拆分。
:::

In [8]:
# ============================================================
# 导入包与读取模拟数据
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm
from statsmodels.iolib.summary2 import summary_col

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# 需要先运行 06_heckman_codes.ipynb，生成该数据文件
df = pd.read_csv("./data/heckman_loan_sim.csv")

print(df.shape)
df.head().round(2)

(3000, 15)


,size,cash,lev,profit,collateral,bank_access,credit_window,loan_latent,loan,rate_latent,rate,firm_size_log,leverage_ratio,cash_ratio,profitability
0,-1.63,1.31,0.31,3.25,-0.35,-0.32,1,0.38,1,5.60,5.60,20.53,0.49,0.30,0.15
1,0.25,0.71,-1.19,-1.42,0.90,1.63,0,1.50,1,6.06,6.06,22.22,0.31,0.26,0.01
2,-1.06,-0.15,-0.16,-0.53,-0.51,1.37,0,-0.23,0,6.85,NaN,21.04,0.43,0.19,0.03
3,0.36,-2.22,0.96,1.03,-1.16,-1.74,1,-2.82,0,7.05,NaN,22.33,0.57,0.02,0.08
4,-0.33,0.63,-0.29,-0.07,-0.58,-0.89,0,-0.42,0,6.39,NaN,21.70,0.41,0.25,0.05


In [9]:
# ============================================================
# 描述性统计：样本选择与贷款利率可观测性
# ============================================================

summary = pd.DataFrame({
    "统计量": ["总样本量", "获得贷款样本量", "获得贷款比例", "贷款利率均值", "贷款利率中位数", "贷款利率标准差"],
    "数值": [
        len(df),
        int(df["loan"].sum()),
        df["loan"].mean(),
        df["rate"].mean(),
        df["rate"].median(),
        df["rate"].std()
    ]
}).round(3)
summary

,统计量,数值
0,总样本量,3000.000
1,获得贷款样本量,1563.000
2,获得贷款比例,0.521
3,贷款利率均值,5.544
4,贷款利率中位数,5.575
5,贷款利率标准差,0.962


In [3]:
# ============================================================
# 获得贷款企业与未获得贷款企业的均值对比
# ============================================================

compare_vars = ["size", "cash", "lev", "profit", "collateral", "bank_access", "credit_window"]
compare_table = df.groupby("loan")[compare_vars].agg(["mean", "std"]).T
compare_table.columns = ["未获得贷款", "获得贷款"]
compare_table.round(3)

未获得贷款   获得贷款
size          mean -0.371  0.341
              std   0.934  0.936
cash          mean -0.090  0.083
              std   0.988  1.004
lev           mean  0.302 -0.277
              std   0.951  0.964
profit        mean -0.170  0.156
              std   0.988  0.985
collateral    mean -0.454  0.417
              std   0.905  0.896
bank_access   mean -0.331  0.304
              std   0.954  0.944
credit_window mean  0.525  0.592
              std   0.500  0.492

![获得贷款企业与未获得贷款企业的特征差异](./figs/limit_dep_heckman_fig02_selected_vs_unselected.png)

上图展示了典型的样本选择结构：获得贷款企业与未获得贷款企业在企业规模、抵押品、银行服务可得性等变量上存在系统差异。若这些差异背后还包含未观测风险，那么只在获得贷款企业中估计贷款利率方程就可能产生选择偏误。

![潜在贷款利率与观测贷款利率](./figs/limit_dep_heckman_fig03_latent_observed_rate.png)

## inverse Mills ratio 的来源与含义

在获得贷款样本中观察到：

$$
y_i = x_i'\beta + u_i,
\quad s_i=1
$$

但 $s_i=1$ 等价于：

$$
s_i^*=z_i'\gamma+v_i>0
$$

即：

$$
v_i>-z_i'\gamma
$$

因此，在被选择样本中：

$$
E(y_i\mid x_i,z_i,s_i=1)
=
x_i'\beta+E(u_i\mid v_i>-z_i'\gamma)
$$

在联合正态假设下，可以得到：

$$
E(u_i\mid v_i>-z_i'\gamma)
=
\rho\sigma_u\lambda(z_i'\gamma)
$$

其中：

$$
\lambda(z_i'\gamma)=\frac{\phi(z_i'\gamma)}{\Phi(z_i'\gamma)}
$$

于是：

$$
E(y_i\mid x_i,z_i,s_i=1)
=
x_i'\beta
+
\rho\sigma_u\lambda(z_i'\gamma)
$$

令 $\theta=\rho\sigma_u$，第二阶段回归可写为：

$$
y_i
=
x_i'\beta
+
\theta\lambda_i
+
\varepsilon_i,
\quad s_i=1
$$

$\lambda_i$ 就是 inverse Mills ratio。它不是普通控制变量，而是由选择方程推导出来的选择修正项。其经济含义是：在给定可观测变量后，一个观测值进入可观测样本的非随机程度。

::: {.callout-important}
### 避坑指南：IMR 的显著性不等于排除变量有效

IMR 显著通常说明选择过程与结果方程误差项存在相关性，但它不能自动证明排他性变量有效。排他性变量是否合理，需要依赖制度背景、经济机制、变量定义和稳健性分析。
:::

In [4]:
# ============================================================
# 手动实现 Heckman 两阶段估计
# ============================================================

# 第一阶段：Probit 选择方程
select_vars = ["size", "cash", "lev", "profit", "collateral", "bank_access", "credit_window"]
Z = sm.add_constant(df[select_vars])
probit_mod = sm.Probit(df["loan"], Z)
probit_res = probit_mod.fit(disp=False)

# 计算 inverse Mills ratio
# 对于 selection event loan = 1，常用 lambda = phi(z gamma) / Phi(z gamma)
xb_select = probit_res.predict(linear=True)
df["imr"] = norm.pdf(xb_select) / np.clip(norm.cdf(xb_select), 1e-8, 1)

# 第二阶段：在获得贷款样本中估计贷款利率方程
outcome_vars = ["size", "cash", "lev", "profit", "collateral"]
df_selected = df.loc[df["loan"] == 1].copy()

X_ols = sm.add_constant(df_selected[outcome_vars])
ols_selected = sm.OLS(df_selected["rate"], X_ols).fit(cov_type="HC1")

X_heck = sm.add_constant(df_selected[outcome_vars + ["imr"]])
heckman_2step = sm.OLS(df_selected["rate"], X_heck).fit(cov_type="HC1")

print("第一阶段 Probit：")
print(probit_res.summary())
print("\n第二阶段 OLS + IMR：")
print(heckman_2step.summary())

第一阶段 Probit：
                          Probit Regression Results                           
Dep. Variable:                   loan   No. Observations:                 3000
Model:                         Probit   Df Residuals:                     2992
Method:                           MLE   Df Model:                            7
Date:                Tue, 28 Apr 2026   Pseudo R-squ.:                  0.3675
Time:                        17:57:00   Log-Likelihood:                -1313.5
converged:                       True   LL-Null:                       -2076.8
Covariance Type:            nonrobust   LLR p-value:                     0.000
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.1804      0.043     -4.167      0.000      -0.265      -0.096
size              0.4795      0.035     13.856      0.000       0.412       0.547
cash              0.1496   

c:\Users\Administrator\.conda\envs\dml050\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)


上面的 Python 两步法有两个作用。第一，它把 Heckman 两阶段估计的逻辑展开为可见步骤：先估计选择方程，再计算 IMR，最后把 IMR 加入结果方程。第二，它为理解 Stata 的 `heckman, twostep` 提供了一个透明版本。

需要注意，手动第二阶段的常规 robust 标准误并不等同于严格的 Heckman 两阶段修正标准误。正式研究中，可以用 Stata 的 `heckman, twostep` 或 `heckman, mle` 作为主要结果。

In [5]:
# ============================================================
# 回归结果表：类似 esttab 的简洁输出
# ============================================================

reg_table = summary_col(
    [ols_selected, heckman_2step],
    model_names=["OLS selected", "Heckman two-step"],
    stars=True,
    float_format="%0.3f",
    info_dict={
        "N": lambda x: f"{int(x.nobs)}",
        "R2": lambda x: f"{x.rsquared:.3f}" if hasattr(x, "rsquared") else ""
    }
)

print(reg_table)


               OLS selected Heckman two-step
--------------------------------------------
const          5.914***     6.191***        
               (0.020)      (0.046)         
size           -0.279***    -0.348***       
               (0.019)      (0.022)         
cash           -0.090***    -0.111***       
               (0.017)      (0.017)         
lev            0.488***     0.546***        
               (0.019)      (0.020)         
profit         -0.361***    -0.411***       
               (0.018)      (0.019)         
collateral     -0.181***    -0.256***       
               (0.021)      (0.024)         
imr                         -0.417***       
                            (0.061)         
R-squared      0.482        0.497           
R-squared Adj. 0.480        0.495           
N              1563         1563            
R2             0.482        0.497           
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01


In [6]:
# ============================================================
# 第一阶段 Probit 的平均边际效应
# ============================================================

mfx = probit_res.get_margeff(at="overall")
print(mfx.summary())

       Probit Marginal Effects       
Dep. Variable:                   loan
Method:                          dydx
At:                           overall
                   dy/dx    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
size              0.1177      0.008     15.361      0.000       0.103       0.133
cash              0.0367      0.007      5.308      0.000       0.023       0.050
lev              -0.0961      0.007    -12.991      0.000      -0.111      -0.082
profit            0.0908      0.007     13.577      0.000       0.078       0.104
collateral        0.1241      0.008     15.657      0.000       0.109       0.140
bank_access       0.1641      0.007     25.099      0.000       0.151       0.177
credit_window     0.1202      0.014      8.381      0.000       0.092       0.148


![Inverse Mills ratio 与贷款利率](./figs/limit_dep_heckman_fig04_imr_rate.png)

![OLS 与 Heckman 两步法的系数对比](./figs/limit_dep_heckman_fig05_ols_heckman_compare.png)

## Stata 实现：两阶段估计与极大似然估计

以下代码可在 Stata 或 nbstata 中执行。若当前工作路径下已有 `./data/heckman_loan_sim.csv`，可直接读取与 Python 相同的模拟数据。

```stata
*------------------------------------------------------------
* 读取模拟数据
*------------------------------------------------------------

clear all
set more off

import delimited "./data/heckman_loan_sim.csv", clear

describe
summarize rate loan size cash lev profit collateral bank_access credit_window
```

```stata
*------------------------------------------------------------
* 基准 OLS：只在获得贷款企业中估计贷款利率方程
*------------------------------------------------------------

reg rate size cash lev profit collateral if loan == 1, vce(robust)
estimates store OLS_selected
```

```stata
*------------------------------------------------------------
* 手动两步法：第一阶段 Probit + 第二阶段加入 IMR
*------------------------------------------------------------

probit loan size cash lev profit collateral bank_access credit_window
predict xb_select, xb

gen imr_manual = normalden(xb_select) / normal(xb_select)

reg rate size cash lev profit collateral imr_manual if loan == 1, ///
    vce(robust)
estimates store Manual_2step
```

```stata
*------------------------------------------------------------
* Stata 官方 Heckman 两阶段估计
*------------------------------------------------------------

heckman rate size cash lev profit collateral, ///
    select(loan = size cash lev profit collateral bank_access credit_window) ///
    twostep first mills(imr_official)

estimates store Heckman_2step
```

```stata
*------------------------------------------------------------
* Stata 官方 Heckman 极大似然估计
*------------------------------------------------------------

heckman rate size cash lev profit collateral, ///
    select(loan = size cash lev profit collateral bank_access credit_window) ///
    mle first vce(robust)

estimates store Heckman_ML
```

```stata
*------------------------------------------------------------
* 使用 esttab 输出结果表
*------------------------------------------------------------

capture which esttab
if _rc{
    ssc install estout, replace
}

esttab OLS_selected Manual_2step Heckman_2step Heckman_ML, ///
    b(%9.3f) se(%9.3f) ///
    star(* 0.10 ** 0.05 *** 0.01) ///
    stats(N, labels("Observations")) ///
    compress
```

Stata 的 `heckman` 同时支持两阶段估计和极大似然估计。`twostep` 指定 Heckman 两阶段估计，`mle` 指定极大似然估计。`mills(newvar)` 可保存 inverse Mills ratio；Stata 文档中也将 `mills()` 和 `nshazard()` 作为同义选项，用于生成 Heckman 所称的 inverse Mills ratio。

## Python 中的直接估计路径

Python 中可以估计 Heckman / Heckit 模型，但生态不如 Stata 的 `heckman` 稳定统一。实际使用时，可以按以下顺序选择工具：

- 若需要课堂演示和透明步骤，使用 `statsmodels` 的 Probit + OLS 手动两步法；
- 若希望调用专门的 Heckit 实现，可尝试 `py4etrics`；
- 若论文结果需要严谨核对，可同时使用 Stata `heckman` 复核；
- 若模型结构更复杂，例如多方程、内生切换或非正态设定，可考虑 Stata 的扩展命令或贝叶斯建模。

`py4etrics` 项目说明中明确包括 Truncated Regression、Tobit Model 和 Heckit Model，安装命令为 `pip install py4etrics`。不过，第三方包的 API 可能随版本变化，使用时应以当前版本文档为准。

In [7]:
# ============================================================
# 可选：检查 py4etrics 是否已经安装
# ============================================================

try:
    import py4etrics
    print("py4etrics 已安装，可以根据当前版本文档调用 Heckit 模型。")
    print("py4etrics 模块信息:", py4etrics)
except ImportError:
    print("尚未安装 py4etrics。可在本地环境中运行：pip install py4etrics")

尚未安装 py4etrics。可在本地环境中运行：pip install py4etrics


::: {.callout-tip}
### 提示词：用 Python 生成 Heckman 两阶段估计代码

我正在分析一个 Heckman 选择模型。结果变量是 `结果变量名`，只有在 `选择变量名=1` 时才可观测。结果方程解释变量包括 `结果方程变量列表`，选择方程解释变量包括 `选择方程变量列表`，其中 `排他性变量名` 只进入选择方程。请帮我生成 Python 代码，完成 Probit 第一阶段、inverse Mills ratio 计算、第二阶段 OLS 回归、结果表整理和基本解释。代码需要包含中文注释，并假定数据保存在 pandas DataFrame `df` 中。
:::

::: {.callout-tip}
### 提示词：用 Stata 生成 Heckman 估计代码

我正在使用 Stata 估计 Heckman 选择模型。结果变量是 `结果变量名`，选择变量是 `选择变量名`，结果方程解释变量包括 `结果方程变量列表`，选择方程解释变量包括 `选择方程变量列表`。其中 `排他性变量名` 只进入选择方程。请帮我生成 `heckman` 的两阶段估计和极大似然估计代码，并使用 `esttab` 输出回归表。代码需要使用 `///` 换行，并添加中文注释。
:::

::: {.callout-tip}
### 提示词：检查排他性变量是否有说服力

我正在为 Heckman 选择模型设计排他性变量。研究问题是 `研究问题`，结果变量是 `结果变量`，选择变量是 `选择变量`，候选排他性变量是 `变量名`。请从经济机制、制度背景、可能的直接效应、需要加入的控制变量和稳健性检验五个角度，评估这个变量是否适合作为选择方程中的排他性变量。
:::

## Stata 扩展命令与模型地图

Heckman selection model 是一个基础框架。实际研究中，结果变量类型和选择机制可能更复杂，可以考虑以下命令或模型：

| 场景 | Stata 命令 | 说明 |
|---|---|---|
| 连续结果变量存在样本选择 | `heckman` | 标准 Heckman selection model |
| 二元结果变量存在样本选择 | `heckprobit` | outcome 为 Probit，带选择机制 |
| 有序结果变量存在样本选择 | `heckoprobit` | outcome 为 ordered probit，带选择机制 |
| 计数结果变量存在样本选择 | `heckpoisson` | outcome 为 Poisson，带选择机制 |
| 内生切换回归 | `movestay` | 不同 regime 下结果方程不同 |
| 多方程混合模型 | `cmp` | 用户扩展命令，适合更复杂的递归多方程结构 |

::: {.callout-important}
### 避坑指南：扩展命令不能替代模型设定

扩展命令可以处理更复杂的因变量类型和方程结构，但不能自动解决识别问题。选择方程和结果方程的经济含义、排他性变量的合理性、样本产生过程的解释，仍然是模型可信度的核心。
:::

## 模型解释：如何写成实证分析语言

在贷款利率案例中，结果可以按如下逻辑解释。

首先，第一阶段 Probit 结果说明哪些因素影响企业进入贷款利率样本。若 `bank_access` 和 `credit_window` 显著为正，说明本地银行服务可得性和政银企对接窗口会提高企业获得银行贷款的概率。这一步不是附属回归，而是在刻画样本如何产生。

其次，第二阶段的结果方程解释贷款利率水平。若 `lev` 的系数为正，说明杠杆率较高的企业面临更高融资成本；若 `profit` 的系数为负，说明盈利能力较强的企业获得更低贷款利率。

再次，IMR 的系数反映选择修正项的重要性。若 IMR 系数显著，说明获得贷款样本不是随机样本，选择过程中的未观测因素与贷款利率方程中的未观测因素相关。若 IMR 系数不显著，则样本选择偏误的证据较弱，但这并不自动证明不存在选择问题。

最后，应将 OLS 与 Heckman 结果并列报告。若核心系数在两类模型中差异较大，说明选择修正对结论有实质影响；若差异较小，则可以把 Heckman 作为稳健性分析的一部分。

## Heckman、Tobit、Truncated regression 和 Two-part model 的区别

| 模型 | 观测机制 | 典型问题 | 关键设定 |
|---|---|---|---|
| Tobit | 潜在结果被删失，边界值仍可观测 | 研发投入大量为 0 | 一个潜在结果方程 |
| Truncated regression | 某些样本根本不进入数据 | 只观察高收入样本 | 截断后的结果方程 |
| Two-part model | 是否参与和正值强度分开 | 是否捐赠与捐赠金额 | 参与方程 + 正值方程 |
| Heckman selection | 结果变量只在被选择样本中可观测 | 只有获得贷款企业有贷款利率 | 选择方程 + 结果方程 |

::: {.callout-important}
### 避坑指南：Heckman 不是 Tobit 的替代品

如果未选择样本的结果变量被记录为 0，问题可能更接近 Tobit 或 Two-part model；如果未选择样本的结果变量根本不可观测，问题才更接近 Heckman selection model。关键不是数据中有没有 0，而是结果变量的观测机制。
:::

## 附录：两阶段估计的推导要点

第一阶段估计选择概率：

$$
P(s_i=1\mid z_i)=\Phi(z_i'\gamma)
$$

得到 $\hat\gamma$ 后，计算：

$$
\hat\lambda_i
=
\frac{\phi(z_i'\hat\gamma)}{\Phi(z_i'\hat\gamma)}
$$

第二阶段在 $s_i=1$ 的样本中估计：

$$
y_i
=
x_i'\beta
+
\theta\hat\lambda_i
+
\varepsilon_i
$$

若 $\theta=\rho\sigma_u$ 显著不为 0，说明选择方程误差项与结果方程误差项相关。若 $\theta=0$，则选择修正项不再发挥作用，被选择样本中的 OLS 与 Heckman 修正结果通常会更接近。

文献中 inverse Mills ratio 有时写成 $\phi(a)/\Phi(a)$，有时写成 $\phi(a)/(1-\Phi(a))$。差异主要来自选择事件的方向写法不同。例如，有的作者把选择事件写成 $v_i>-z_i'\gamma$，有的写成非选择事件。实际使用时，应保持选择事件、公式写法和软件定义一致。

## 参考文献

- Heckman, J. J. (1979). Sample selection bias as a specification error. Econometrica, 47(1), 153–161. [Link](https://doi.org/10.2307/1912352), [PDF](http://sci-hub.ren/10.2307/1912352), [Google](https://scholar.google.com/scholar?q=Sample+selection+bias+as+a+specification+error).

- Mroz, T. A. (1987). The sensitivity of an empirical model of married women’s hours of work to economic and statistical assumptions. Econometrica, 55(4), 765–799. [Link](https://doi.org/10.2307/1911029), [PDF](http://sci-hub.ren/10.2307/1911029), [Google](https://scholar.google.com/scholar?q=The+sensitivity+of+an+empirical+model+of+married+women%27s+hours+of+work+to+economic+and+statistical+assumptions).

- Stiglitz, J. E., & Weiss, A. (1981). Credit rationing in markets with imperfect information. American Economic Review, 71(3), 393–410. [Link](https://www.jstor.org/stable/1802787), [PDF](https://academiccommons.columbia.edu/doi/10.7916/D8V12FT1), [Google](https://scholar.google.com/scholar?q=Credit+rationing+in+markets+with+imperfect+information).

- Petersen, M. A., & Rajan, R. G. (1994). The benefits of lending relationships: Evidence from small business data. The Journal of Finance, 49(1), 3–37. [Link](https://doi.org/10.1111/j.1540-6261.1994.tb04418.x), [PDF](http://sci-hub.ren/10.1111/j.1540-6261.1994.tb04418.x), [Google](https://scholar.google.com/scholar?q=The+benefits+of+lending+relationships+evidence+from+small+business+data).

- Vella, F. (1998). Estimating models with sample selection bias: A survey. Journal of Human Resources, 33(1), 127–169. [Link](https://doi.org/10.2307/146317), [PDF](http://sci-hub.ren/10.2307/146317), [Google](https://scholar.google.com/scholar?q=Estimating+models+with+sample+selection+bias+A+survey).